In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [6]:

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph import DFGrepInterference, DFGrepWorkflow 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage_pegasus_1_deg" 

condition_fn = None #

if app_name == "montage_pegasus_1_deg":
    filename = "/usr/workspace/pandey2/dlp_logs/montage_pegasus_1_deg/*pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage":
    filename ="/usr/workspace/pandey2/dlp_logs/montage_16_48ppn/montage*.pfw"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m2d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-2-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "montage2m7d":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/montage-2mass-7-degree/*.pfw.gz"
    cp_dir = "/p/lustre2/pandey2/cp_dir/montage"

elif app_name == "deepspeed":
    filename = "/usr/workspace/iopp/dlp_traces/deepspeed_8_4ppn/*.pfw.gz"

else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, 
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=False, 
                                host_pattern=r'lassen(\d+)', time_granularity=30e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [07:41:14] Initialized Client with 768 workers and link http://134.9.71.27:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:678]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [07:41:34] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:670]


2024-10-02 12:25:03,943 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [8]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [9]:
def montage_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["filename"] = str(json_object["args"]["fname"])   
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))    
    return d

load_cols_montage = {'filename':"string[pyarrow]",'mount_point':"string[pyarrow]"}


In [10]:
analyzer_montage = DFAnalyzer(filename,load_fn=montage_cols_function, load_cols=load_cols_montage, load_data={"mount_point":trie})

[INFO] [07:41:52] Created index for 1657 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:376]
[INFO] [07:41:52] Total size of all files are <dask.bag.core.Item object at 0x1554aa7572b0> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:378]
[INFO] [07:42:00] Loading 1742 batches out of 1657 files and has 4742072 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:391]
[INFO] [07:42:12] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:436]
[INFO] [07:42:12] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:442]


In [11]:
analyzer_montage.events.head()

,name,cat,pid,tid,ts,te,dur,tinterval,trange,hostname,compute_time,io_time,app_io_time,total_time,filename,phase,size,mount_point
0,start,dftracer,558673,1117346,43956,43956,0,<NA>,0,corona240,<NA>,<NA>,<NA>,0,<NA>,0,<NA>,<NA>
1,readlink,POSIX,558673,1117346,49316,49323,7,<NA>,0,corona240,<NA>,7,<NA>,7,/usr/bin/python3,2,<NA>,/usr/bin
2,readlink,POSIX,558673,1117346,49352,49356,4,<NA>,0,corona240,<NA>,4,<NA>,4,/etc/alternatives/python3,2,<NA>,/etc/alternatives
3,readlink,POSIX,558673,1117346,49369,49373,4,<NA>,0,corona240,<NA>,4,<NA>,4,/usr/bin/python3.6,2,<NA>,/usr/bin
4,readlink,POSIX,558673,1117346,49385,49388,3,<NA>,0,corona240,<NA>,3,<NA>,3,/usr/libexec/platform-python3.6,2,<NA>,/usr/libexec


In [8]:
analyzer_montage.events['id'] = analyzer_montage.events.index

In [14]:

# eventsDF = analyzer_montage.events[analyzer_montage.events['cat'] == "POSIX" ] # only posix events

In [9]:
cp_dir

'/p/lustre2/pandey2/cp_dir/montage'

In [10]:
IFCalculator = DFGrepInterference(analyzer_montage.events, app_name=app_name, cp_dir=cp_dir, existing=False)

In [13]:
analyzer_montage.events.compute()

KeyError: 'cat'

In [11]:
IFCalculator.get_degree()
IFCalculator.get_interference()
IFCalculator.get_interference_metadata()

KeyError: 'cat'

In [10]:
IFCalculator.write_checkpoint("inter", cp_dir=cp_dir)